In [1]:
from dance.transforms import Compose
from dance.datasets.spatial import SpatialLIBDDataset
from dance.modules.spatial.spatial_domain.spagcn import SpaGCN
from dance.utils import set_seed
from dance.transforms import CellPCA, Compose, FilterGenesMatch, SetConfig
from dance.transforms.graph import SpaGCNGraph, SpaGCNGraph2D

/mnt/g/Projects/dance/dance/utils/matrix.py:178: NumbaExperimentalFeatureWarning: First-class function type feature is experimental
  for j in numba.prange(n):


In [2]:
import scanpy as sc
import numpy as np
import gc
import pandas as pd

In [3]:
from dance.modules.spatial.spatial_domain.stagate import Stagate

In [4]:
import sklearn.metrics

In [5]:
from tqdm import tqdm

In [6]:
from dance import logger
logger.setLevel("ERROR")

In [7]:
import time


In [8]:
from dance.transforms import AnnDataTransform, Compose, SetConfig
from dance.transforms.graph import StagateGraph
from dance.typing import Any, LogLevel, Optional, Tuple

# radius doesn't matter with "knn" specified.
def preprocessing_pipeline(model_name: str = "knn", radius: float = 150, n_neighbors: int = 6, 
                           log_level: LogLevel = "ERROR"):
        return Compose(
            AnnDataTransform(sc.pp.normalize_total, target_sum=1e4),
            AnnDataTransform(sc.pp.log1p),
            StagateGraph(model_name, radius=radius, n_neighbors=n_neighbors),
            SetConfig({
                "feature_channel": "StagateGraph",
                "feature_channel_type": "obsp",
                "label_channel": "label",
                "label_channel_type": "obs"
            }),
            log_level=log_level,
        )

preprocessing_pipeline = preprocessing_pipeline()

In [10]:
from dance.datasets.base import Data
aris = []
nmis = []
t0 = time.time()
adata = sc.read_h5ad("../../SEDR_analyses/data/BRCA1/adata_raw.h5ad")
adata.X = adata.X.todense()
scale_factor = adata.uns["spatial"]["V1_Breast_Cancer_Block_A_Section_1"]["scalefactors"]["tissue_hires_scalef"]
adata.obsm['spatial_pixel'] = adata.obsm['spatial'] * scale_factor 
# The scale_factor doesn't really matter because preprocessing_pipeline uses knn, but visium data has it so we use it.
adata.obs['label'] = 0

data = Data(adata)
preprocessing_pipeline(data)

adj, y = data.get_data(return_type="default")
x = data.data.X
edge_list_array = np.vstack(np.nonzero(adj))

model = Stagate(hidden_dims=[x.shape[1], 512, 32])
pred = model.fit_predict((x, edge_list_array), 1000, num_cluster=20, random_state=0)
adata.obs['stagate'] = pred
adata.obs[['stagate']].to_csv(f"../../Steamboat/revised/saved_results/visium_stagate_spatial_domain.csv")

# ari = sklearn.metrics.adjusted_rand_score(adata.obs['parcellation_division'], adata.obs['stagate'])
# nmi = sklearn.metrics.normalized_mutual_info_score(adata.obs['parcellation_division'], adata.obs['stagate'])

# print(i, adata.obs['stagate'].nunique(), ari, nmi, f"{(time.time() - t0) / 60:.2f} minutes")
# aris.append(ari)
# nmis.append(nmi)

# del adata
# gc.collect()

tt = time.time() - t0

/home/lshh/miniconda3/envs/dance/lib/python3.11/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/lshh/miniconda3/envs/dance/lib/python3.11/site-packages/anndata/_core/storage.py:85: ImplicitModificationWarning: X should not be a np.matrix, use np.ndarray instead.
  warnings.warn(msg, ImplicitModificationWarning)
100%|███████████████████████████████████████████████████████████████████████████████| 1000/1000 [02:00<00:00,  8.31it/s]


In [11]:
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
labels_true = adata.obs['fine_annot_type'].values
labels_pred = adata.obs['stagate'].values
n_clusters = adata.obs['stagate'].nunique()
nmi = normalized_mutual_info_score(labels_true, labels_pred)
ari = adjusted_rand_score(labels_true, labels_pred)
print(f"NMI: {nmi:.4f}, ARI: {ari:.4f}, n_clusters: {n_clusters}")

NMI: 0.6819, ARI: 0.5742, n_clusters: 20


In [13]:
import json
with open(f"../../Steamboat/revised/saved_results/visium_stagate_spatial_domain_scores.json", "w") as file: 
        json.dump({"ARI": ari, "NMI": nmi, "n_clutsers": n_clusters}, file)